In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

# 1. Load dataset
df = pd.read_csv("titanic.csv")

# Remove duplicate column names, keeping the first occurrence
df = df.loc[:, ~df.columns.duplicated()]

print("First 5 records:")
print(df.head())

# 2. Select relevant features and target
X = df[["Pclass", "Sex", "Age", "Fare"]]
y = df["Survived"]

# 3. Handle missing values
X = X.copy()
X["Age"] = X["Age"].fillna(X["Age"].median())
X["Fare"] = X["Fare"].fillna(X["Fare"].median())
X["Sex"] = X["Sex"].fillna(X["Sex"].mode()[0])

# Encode Sex: male = 0, female = 1
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

# 4. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42
)

rf_model.fit(X_train, y_train)

# 6. Make predictions
y_pred = rf_model.predict(X_test)

print("\nActual vs Predicted:")
print(pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
}).head())

# 7. Evaluate model
print("\nModel Evaluation:")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 8. Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Survived", "Survived"]
)

disp.plot(cmap="Blues")
plt.title("Titanic Survival Confusion Matrix")
plt.show()
